Important Libraries

In [1]:
import mysql.connector
import re
from datetime import datetime
import csv

Database Connection

In [2]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root123",
    database="finance_master",
    auth_plugin="mysql_native_password"
)

cursor = conn.cursor()

Input Validation Functions

In [3]:
def validate_email(email):
    return re.match(r'^[\w\.-]+@[\w\.-]+\.\w+$',email)

def validate_employee_email(email):
    return re.match(r'^[\w\.-]+@finance\.com$',email)

def validate_password(password):
    return len(password)>=6 and any(c.isdigit() for c in password)

def validate_date(date):
    try:
        datetime.strptime(date,"%Y-%m-%d")
        return True
    except:
        return False

def validate_positive(v):
    try:
        x=float(v)
        return x>0
    except:
        return False

def safe_float(msg):
    val=input(msg)
    if not validate_positive(val):
        print("Invalid value")
        return None
    return float(val)

Customer Registration and Login

In [4]:
def register():
    name=input("Name: ")
    email=input("Email: ")
    password=input("Password: ")

    if not validate_email(email):
        print("Invalid email")
        return

    if not validate_password(password):
        print("Password must contain number and 6 chars")
        return

    cursor.execute("SELECT * FROM users WHERE email=%s",(email,))
    if cursor.fetchone():
        print("Email already exists")
        return

    cursor.execute("INSERT INTO users(name,email,password) VALUES(%s,%s,%s)",(name,email,password))
    conn.commit()
    print("Registration successful")

def login():
    email=input("Email: ")
    password=input("Password: ")

    cursor.execute("SELECT user_id FROM users WHERE email=%s AND password=%s",(email,password))
    r=cursor.fetchone()

    if r:
        print("Login successful")
        return r[0]
    else:
        print("Invalid credentials")
        return None

Employee Login System

In [5]:
def employee_login():
    email=input("Employee email: ")
    password=input("Password: ")

    if not validate_employee_email(email):
        print("Invalid company email")
        return None

    cursor.execute("SELECT emp_id FROM employees WHERE email=%s AND password=%s",(email,password))
    r=cursor.fetchone()

    if r:
        print("Employee login successful")
        return r[0]
    else:
        print("Invalid credentials")
        return None

Income and Expense Management

In [6]:
def add_income(uid):
    source=input("Income source: ")
    amount=safe_float("Amount: ")
    date=input("Date YYYY-MM-DD: ")

    if not amount or not validate_date(date):
        print("Invalid data")
        return

    cursor.execute("INSERT INTO income(user_id,source,amount,date) VALUES(%s,%s,%s,%s)",(uid,source,amount,date))
    conn.commit()
    print("Income added")

def add_expense(uid):
    category=input("Category: ")
    amount=safe_float("Amount: ")
    date=input("Date YYYY-MM-DD: ")

    if not amount or not validate_date(date):
        print("Invalid data")
        return

    cursor.execute("INSERT INTO expenses(user_id,category,amount,date) VALUES(%s,%s,%s,%s)",(uid,category,amount,date))
    conn.commit()

    cursor.execute("SELECT monthly_limit FROM budget WHERE user_id=%s AND category=%s",(uid,category))
    limit=cursor.fetchone()

    if limit:
        cursor.execute("SELECT SUM(amount) FROM expenses WHERE user_id=%s AND category=%s",(uid,category))
        total=cursor.fetchone()[0] or 0
        if total>limit[0]:
            print("Warning: Budget exceeded")

    print("Expense added")

Financial Summary

In [7]:
def financial_summary(uid):
    cursor.execute("SELECT SUM(amount) FROM income WHERE user_id=%s",(uid,))
    income=cursor.fetchone()[0] or 0

    cursor.execute("SELECT SUM(amount) FROM expenses WHERE user_id=%s",(uid,))
    expense=cursor.fetchone()[0] or 0

    savings=income-expense
    rate=(savings/income*100) if income>0 else 0

    print("Income:",income)
    print("Expense:",expense)
    print("Savings:",savings)
    print("Savings rate:",round(rate,2),"%")

Investment Portfolio Management

In [8]:
def add_investment(uid):
    t=input("Investment type: ")
    name=input("Name: ")
    invested=safe_float("Invested amount: ")
    current=safe_float("Current value: ")

    if not invested or not current:
        return

    cursor.execute("INSERT INTO investments(user_id,type,name,invested,current_value) VALUES(%s,%s,%s,%s,%s)",
                   (uid,t,name,invested,current))
    conn.commit()
    print("Investment added")

def portfolio(uid):
    cursor.execute("SELECT SUM(invested),SUM(current_value) FROM investments WHERE user_id=%s",(uid,))
    data=cursor.fetchone()

    invested=data[0] or 0
    current=data[1] or 0

    profit=current-invested
    roi=(profit/invested*100) if invested>0 else 0

    print("Total invested:",invested)
    print("Current value:",current)
    print("Profit:",profit)
    print("ROI:",round(roi,2),"%")

Financial Goals Tracking

In [9]:
def add_goal(uid):
    name=input("Goal name: ")
    target=safe_float("Target amount: ")
    saved=safe_float("Saved amount: ")

    if not target or not saved:
        return

    cursor.execute("INSERT INTO goals(user_id,goal_name,target_amount,saved_amount) VALUES(%s,%s,%s,%s)",
                   (uid,name,target,saved))
    conn.commit()
    print("Goal added")

def goal_progress(uid):
    cursor.execute("SELECT goal_name,target_amount,saved_amount FROM goals WHERE user_id=%s",(uid,))
    goals=cursor.fetchall()

    for g in goals:
        progress=(g[2]/g[1]*100) if g[1]>0 else 0
        print(g[0],round(progress,2),"% completed")

Financial Calculators (SIP & CAGR)

In [10]:
def sip():
    monthly=safe_float("Monthly investment: ")
    rate=safe_float("Return %: ")
    years=int(input("Years: "))

    if not monthly or not rate or years<=0:
        print("Invalid input")
        return

    r=rate/(12*100)
    months=years*12
    fv=monthly*(((1+r)**months-1)/r)*(1+r)

    print("Future value:",round(fv,2))

def cagr():
    initial=safe_float("Initial investment: ")
    final=safe_float("Final value: ")
    years=int(input("Years: "))

    if not initial or not final or years<=0:
        print("Invalid input")
        return

    c=((final/initial)**(1/years)-1)*100
    print("CAGR:",round(c,2),"%")

Net Worth and Tax Estimation

In [11]:
def net_worth(uid):
    cursor.execute("SELECT SUM(current_value) FROM investments WHERE user_id=%s",(uid,))
    assets=cursor.fetchone()[0] or 0

    cursor.execute("SELECT SUM(amount) FROM expenses WHERE user_id=%s",(uid,))
    liabilities=cursor.fetchone()[0] or 0

    print("Net worth:",assets-liabilities)

def tax(uid):
    cursor.execute("SELECT SUM(amount) FROM income WHERE user_id=%s",(uid,))
    income=cursor.fetchone()[0] or 0

    if income<=250000:
        t=0
    elif income<=500000:
        t=income*0.05
    elif income<=1000000:
        t=income*0.1
    else:
        t=income*0.2

    print("Estimated tax:",round(t,2))

Employee Management Functions

In [12]:
def view_all_customers():
    cursor.execute("SELECT user_id,name,email FROM users")
    data=cursor.fetchall()
    for d in data:
        print(d)

def verify_customer():
    uid=input("Customer ID: ")
    cursor.execute("SELECT name,email FROM users WHERE user_id=%s",(uid,))
    data=cursor.fetchone()
    if data:
        print("Customer:",data)
    else:
        print("Customer not found")

Customer Transactions

In [13]:
def view_customer_transactions():
    uid = input("Enter Customer ID: ")

    print("\nINCOME RECORDS")
    cursor.execute("SELECT source, amount, date FROM income WHERE user_id=%s", (uid,))
    income = cursor.fetchall()

    if income:
        for i in income:
            print("Source:", i[0], "| Amount:", i[1], "| Date:", i[2])
    else:
        print("No income records")

    print("\nEXPENSE RECORDS")
    cursor.execute("SELECT category, amount, date FROM expenses WHERE user_id=%s", (uid,))
    expense = cursor.fetchall()

    if expense:
        for e in expense:
            print("Category:", e[0], "| Amount:", e[1], "| Date:", e[2])
    else:
        print("No expense records")

Customer Summary

In [14]:
def employee_summary():
    uid = input("Enter Customer ID: ")

    cursor.execute("SELECT SUM(amount) FROM income WHERE user_id=%s", (uid,))
    income = cursor.fetchone()[0] or 0

    cursor.execute("SELECT SUM(amount) FROM expenses WHERE user_id=%s", (uid,))
    expense = cursor.fetchone()[0] or 0

    savings = income - expense

    print("\nCUSTOMER FINANCIAL SUMMARY")
    print("Total Income:", income)
    print("Total Expenses:", expense)
    print("Savings:", savings)

Customer Budget

In [15]:
def set_customer_budget():
    uid = input("Enter Customer ID: ")
    category = input("Enter Expense Category: ")
    limit = safe_float("Enter Monthly Budget: ")

    if not limit:
        return

    cursor.execute(
        "INSERT INTO budget(user_id, category, monthly_limit) VALUES(%s,%s,%s)",
        (uid, category, limit)
    )

    conn.commit()
    print("Budget set successfully")

Data Export Feature

In [16]:
def export_customers():
    cursor.execute("SELECT user_id, name, email FROM users")
    data = cursor.fetchall()

    with open("customers_export.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["UserID", "Name", "Email"])

        for d in data:
            writer.writerow(d)

    print("Customers exported to customers_export.csv")

Employee Panel Menu

In [17]:
def employee_menu(emp):
    while True:
        print("\nEMPLOYEE PANEL")
        print("1 Verify customer")
        print("2 View all customers")
        print("3 View transactions")
        print("4 Customer summary")
        print("5 Set customer budget")
        print("6 Export customers")
        print("7 Exit")

        try:
            c = int(input("Choice: "))
        except:
            print("Invalid input")
            continue

        if c == 1:
            verify_customer()

        elif c == 2:
            view_all_customers()

        elif c == 3:
            view_customer_transactions()

        elif c == 4:
            employee_summary()

        elif c == 5:
            set_customer_budget()

        elif c == 6:
            export_customers()

        elif c == 7:
            break

        else:
            print("Invalid choice")

Customer Panel Menu

In [18]:
def customer_menu(uid):
    while True:
        print("\nCUSTOMER PANEL")
        print("1 Add income")
        print("2 Add expense")
        print("3 Financial summary")
        print("4 Add investment")
        print("5 Portfolio")
        print("6 Add goal")
        print("7 Goal progress")
        print("8 SIP calculator")
        print("9 CAGR calculator")
        print("10 Net worth")
        print("11 Tax estimator")
        print("12 Exit")

        try:
            ch=int(input("Choice: "))
        except:
            print("Invalid input")
            continue

        if ch==1:
            add_income(uid)

        elif ch==2:
            add_expense(uid)

        elif ch==3:
            financial_summary(uid)

        elif ch==4:
            add_investment(uid)

        elif ch==5:
            portfolio(uid)

        elif ch==6:
            add_goal(uid)

        elif ch==7:
            goal_progress(uid)

        elif ch==8:
            sip()

        elif ch==9:
            cagr()

        elif ch==10:
            net_worth(uid)

        elif ch==11:
            tax(uid)

        elif ch==12:
            break

        else:
            print("Invalid choice")

Main System Execution

In [19]:
def system():
    while True:
        print("\nFINANCE MANAGEMENT SYSTEM")
        print("1 Customer register")
        print("2 Customer login")
        print("3 Employee login")
        print("4 Exit")

        try:
            c=int(input("Choice: "))
        except:
            print("Invalid input")
            continue

        if c==1:
            register()
        elif c==2:
            uid=login()
            if uid:
                customer_menu(uid)
        elif c==3:
            emp=employee_login()
            if emp:
                employee_menu(emp)
        elif c==4:
            break

system()


FINANCE MANAGEMENT SYSTEM
1 Customer register
2 Customer login
3 Employee login
4 Exit


Choice:  4
